# Notebook 06: Ray Train

## Why This Matters

Ray Train provides a high-level API for distributed training that handles the boilerplate of `init_process_group`, data sharding, checkpoint saving, and fault tolerance. It works with PyTorch, TensorFlow, Hugging Face, and Lightning. The key value: you write a single-worker training loop, and Ray Train scales it to multiple nodes without changing your code. This is the abstraction layer used in production at organizations that need to run hundreds of concurrent training experiments.


In [ ]:
%pip install -q 'ray[train]' torch numpy

In [ ]:
import ray
from ray import train
from ray.train import ScalingConfig, Checkpoint
from ray.train.torch import TorchTrainer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os

if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, ignore_reinit_error=True)
print(f"Ray Train version: {ray.__version__}")
print("Ready.")

## 1. The Train Loop Per Worker

Ray Train's `TorchTrainer` runs a **train_loop_per_worker** function on each worker. This function:
1. Gets the world rank and size from `ray.train.get_context()`
2. Wraps the model with DDP (done automatically by `ray.train.torch.prepare_model`)
3. Wraps the dataloader with a distributed sampler (done by `prepare_data_loader`)
4. Reports metrics via `ray.train.report()`
5. Saves checkpoints via `ray.train.report(checkpoint=...)`

Key advantage: you do not call `init_process_group` or `DistributedSampler` -- Ray handles this.


In [ ]:
# Define the per-worker training loop
def train_loop_per_worker(config):
    # Hyperparameters from config
    lr = config.get('lr', 1e-3)
    n_epochs = config.get('n_epochs', 5)
    batch_size = config.get('batch_size', 64)
    
    # Build model
    model = nn.Sequential(
        nn.Linear(32, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 10)
    )
    
    # Ray Train wraps model with DDP automatically
    model = train.torch.prepare_model(model)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    # Synthetic dataset
    torch.manual_seed(42)
    X = torch.randn(1000, 32)
    Y = torch.randint(0, 10, (1000,))
    dataset = TensorDataset(X, Y)
    
    # Ray Train wraps DataLoader with DistributedSampler
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    dataloader = train.torch.prepare_data_loader(dataloader)
    
    # Training loop
    for epoch in range(n_epochs):
        model.train()
        epoch_losses = []
        for x_batch, y_batch in dataloader:
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        
        mean_loss = np.mean(epoch_losses)
        
        # Report metrics and checkpoint
        checkpoint = None
        if epoch == n_epochs - 1:
            import tempfile
            with tempfile.TemporaryDirectory() as tmpdir:
                torch.save(model.state_dict(), os.path.join(tmpdir, 'model.pt'))
                checkpoint = Checkpoint.from_directory(tmpdir)
        
        train.report({'loss': mean_loss, 'epoch': epoch}, checkpoint=checkpoint)

# Configure scaling
scaling_config = ScalingConfig(
    num_workers=2,          # 2 DDP workers
    use_gpu=torch.cuda.is_available(),
    resources_per_worker={'CPU': 1, 'GPU': 1 if torch.cuda.is_available() else 0}
)

# Create and run trainer
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={'lr': 1e-3, 'n_epochs': 3, 'batch_size': 64},
    scaling_config=scaling_config,
)

result = trainer.fit()
print(f'Final metrics: {result.metrics}')
print(f'Checkpoint: {result.checkpoint}')

## 2. Fault Tolerance and Checkpointing

Ray Train's major advantage over raw DDP: **automatic fault tolerance**. If a worker fails mid-training, Ray Train can restart it from the last checkpoint.

The checkpoint API:
- `Checkpoint.from_directory(path)`: create checkpoint from a local directory
- `train.get_checkpoint()`: inside the loop, get the latest checkpoint to resume from
- `result.checkpoint`: the final checkpoint from `trainer.fit()`


In [ ]:
# Fault-tolerant training loop with checkpoint resume
def fault_tolerant_loop(config):
    model = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 2))
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    start_epoch = 0
    
    # Resume from checkpoint if available
    checkpoint = train.get_checkpoint()
    if checkpoint:
        with checkpoint.as_directory() as ckpt_dir:
            state = torch.load(os.path.join(ckpt_dir, 'state.pt'))
            model.load_state_dict(state['model'])
            optimizer.load_state_dict(state['optimizer'])
            start_epoch = state['epoch'] + 1
        print(f'Resumed from epoch {start_epoch}')
    
    model = train.torch.prepare_model(model)
    
    X = torch.randn(200, 16)
    Y = torch.randint(0, 2, (200,))
    dataset = TensorDataset(X, Y)
    dataloader = DataLoader(dataset, batch_size=32)
    dataloader = train.torch.prepare_data_loader(dataloader)
    
    for epoch in range(start_epoch, config['n_epochs']):
        losses = []
        for x, y in dataloader:
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        
        import tempfile
        with tempfile.TemporaryDirectory() as tmpdir:
            torch.save({
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'epoch': epoch
            }, os.path.join(tmpdir, 'state.pt'))
            ckpt = Checkpoint.from_directory(tmpdir)
            train.report({'loss': np.mean(losses), 'epoch': epoch}, checkpoint=ckpt)

trainer = TorchTrainer(
    train_loop_per_worker=fault_tolerant_loop,
    train_loop_config={'lr': 1e-3, 'n_epochs': 3},
    scaling_config=ScalingConfig(num_workers=1)
)
result = trainer.fit()
print('Fault-tolerant training complete.')
print(f'Final loss: {result.metrics["loss"]:.4f}')

## 3. Comparison: Ray Train vs PyTorch Lightning vs Hugging Face Trainer

| Feature | Raw DDP | Ray Train | PyTorch Lightning | HF Trainer |
|---------|---------|-----------|-------------------|------------|
| Init complexity | High | Low | Low | Very Low |
| Multi-node | Manual | Automatic | Manual setup | Automatic |
| Fault tolerance | No | Yes | Partial | No |
| HPO integration | No | Native (Ray Tune) | Partial | No |
| Custom training | Full control | Full control | Some constraints | Limited |
| Ecosystem | PyTorch | ML/Ray stack | PyTorch | HuggingFace |

**Choose Ray Train when**: you need fault tolerance, you're already using Ray, you want native HPO integration, or you have heterogeneous workers.

**Choose Lightning when**: you want structured, opinionated training loops with good logging integrations.

**Choose HF Trainer when**: you're fine-tuning a HuggingFace model and want zero-config training.


## Summary

### Key Concepts

| Concept | API | Description |
|---------|-----|-------------|
| Per-worker loop | `train_loop_per_worker(config)` | Function running on each worker |
| DDP wrapping | `train.torch.prepare_model(model)` | Automatic DDP wrap |
| Distributed dataloader | `train.torch.prepare_data_loader(dl)` | Adds DistributedSampler |
| Report metrics | `train.report({'loss': loss})` | Send metrics to driver |
| Checkpoint | `Checkpoint.from_directory(path)` | Save state for fault tolerance |
| Resume | `train.get_checkpoint()` | Get latest checkpoint on resume |

**Next**: `07_ray_tune.ipynb` -- hyperparameter search at scale with ASHA and PBT.
